In [1]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import numpy as np
import time
import pickle
import os
import pandas as pd
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

In [2]:
# import training and test data
data = open("data/splits_full_scaled.pkl", "rb")
[X_train, X_test, y_train, y_test] = pickle.load(data)
data.close()

In [4]:
class MLP(eqx.Module):
    layers: list
    dropout: eqx.nn.Dropout

    def __init__(self, key, sizes, dropout_rate=0.3):
        keys = jax.random.split(key, len(sizes))
        self.dropout = eqx.nn.Dropout(p=dropout_rate)
        self.layers = [
            eqx.nn.Linear(sizes[i], sizes[i + 1], key=keys[i + 1])
            for i in range(len(sizes) - 1)
        ]

    def __call__(self, x, key=None, mode="train"):
        for i, layer in enumerate(self.layers[:-1]):
            x = jax.nn.relu(layer(x))
            if mode == "train" and key is not None:
                subkey = jax.random.fold_in(key, i)
                x = self.dropout(x, key=subkey)
        return self.layers[-1](x)


# ----------------------------
# 3. Training und Auswertung
# ----------------------------


def compute_loss(model, x, y, key):
    logits = jax.vmap(lambda x_i: model(x_i, key, mode="train"))(x)
    labels = jax.nn.one_hot(y, num_classes=3)
    return optax.softmax_cross_entropy(logits, labels).mean()


def accuracy(model, x, y):
    preds = jnp.argmax(jax.vmap(lambda x_i: model(x_i, mode="eval"))(x), axis=1)
    return jnp.mean(preds == y)


def moving_average(x, window=5):
    return np.convolve(x, np.ones(window) / window, mode="valid")


def train_model(
    X_train,
    y_train,
    X_test,
    y_test,
    epochs=300,
    lr=1e-3,
    dropout_rate=0.3,
    hidden_sizes=[128, 64],
):
    key = jax.random.PRNGKey(0)
    model = MLP(key, [X_train.shape[1]] + hidden_sizes + [3], dropout_rate=dropout_rate)

    optimizer = optax.adamw(learning_rate=lr, weight_decay=1e-4)
    opt_state = optimizer.init(model)

    train_losses = []
    test_accuracies = []
    best_model = model
    best_acc = -1
    best_epoch = 0

    @eqx.filter_value_and_grad
    def loss_fn(model, x, y, key):
        return compute_loss(model, x, y, key)

    for epoch in range(epochs):
        epoch_key = jax.random.fold_in(key, epoch)
        loss, grads = loss_fn(model, X_train, y_train, epoch_key)
        updates, opt_state = optimizer.update(grads, opt_state, model)
        model = eqx.apply_updates(model, updates)

        acc = accuracy(model, X_test, y_test)
        train_losses.append(float(loss))
        test_accuracies.append(float(acc))

        if acc > best_acc:
            best_acc = acc
            best_model = model
            best_epoch = epoch

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d}: Loss={loss:.4f}, Test Acc={acc:.4f}")

    print(f"Best Epoch: {best_epoch} → Test Accuracy: {best_acc:.4f}")
    return best_model, train_losses, test_accuracies


# Trainieren
model, losses, accuracies = train_model(
    jnp.array(X_train),
    jnp.array(y_train),
    jnp.array(X_test),
    jnp.array(y_test),
    epochs=200,
    lr=best["lr"],
    hidden_sizes=best["hidden"],
    dropout_rate=best["dropout"],
)

results_df = pd.DataFrame(results).sort_values(by="acc", ascending=False)
best = results_df.iloc[0]


# Gleitende Mittelwert-Glättung
smoothed_acc = moving_average(accuracies, window=10)
best_epoch = np.argmax(smoothed_acc) + 5  # +5 da 'valid'-convolve den Start verschiebt
best_acc = accuracies[best_epoch]
best_loss = losses[best_epoch]
print(f"\nBest test accuracy (smoothed) at epoch {best_epoch}: {best_acc:.4f}")

# ----------------------------
# 4. Plot: Loss & Accuracy Verlauf
# ----------------------------

plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(losses, label="Loss")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best epoch ({best_epoch})")
plt.title("Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(accuracies, label="Test Accuracy", alpha=0.5)
plt.plot(
    range(5, len(smoothed_acc) + 5),
    smoothed_acc,
    label="Smoothed Accuracy",
    color="green",
)
plt.axvline(best_epoch, color="red", linestyle="--")
plt.title("Test Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

NameError: name 'best' is not defined

In [5]:
def grid_search_tuning(X_train, y_train, X_test, y_test, param_grid, epochs=100):
    results = []
    for dropout in param_grid["dropout_rates"]:
        for lr in param_grid["learning_rates"]:
            for hidden in param_grid["hidden_sizes"]:
                print(f"\nTesting: dropout={dropout}, lr={lr}, hidden={hidden}")

                model, _, accs = train_model(
                    jnp.array(X_train),
                    jnp.array(y_train),
                    jnp.array(X_test),
                    jnp.array(y_test),
                    epochs=epochs,
                    lr=lr,
                    hidden_sizes=hidden,
                    dropout_rate=dropout,
                )

                final_acc = accs[-1]
                results.append(
                    {"dropout": dropout, "lr": lr, "hidden": hidden, "acc": final_acc}
                )
                print(f"Final Test Accuracy: {final_acc:.4f}")
    return results


# Define hyperparameter ranges
param_grid = {
    "dropout_rates": [0.1, 0.3, 0.5],
    "learning_rates": [1e-4, 5e-4, 1e-3],
    "hidden_sizes": [[64, 32], [128, 64], [256, 128]],
}

# Run grid search
results = grid_search_tuning(X_train, y_train, X_test, y_test, param_grid, epochs=100)

# Show results
results_df = pd.DataFrame(results).sort_values(by="acc", ascending=False)
display(results_df)

# Train best model again (optional)
best = results_df.iloc[0]
print("Best configuration:", best)

final_model, _, _ = train_model(
    jnp.array(X_train),
    jnp.array(y_train),
    jnp.array(X_test),
    jnp.array(y_test),
    epochs=150,
    lr=best["lr"],
    hidden_sizes=best["hidden"],
    dropout_rate=best["dropout"],
)


Testing: dropout=0.1, lr=0.0001, hidden=[64, 32]
Epoch   0: Loss=1.1434, Test Acc=0.2055
Epoch  10: Loss=0.9808, Test Acc=0.5525
Epoch  20: Loss=0.9410, Test Acc=0.5525
Epoch  30: Loss=0.8993, Test Acc=0.5525
Epoch  40: Loss=0.8564, Test Acc=0.5525
Epoch  50: Loss=0.8349, Test Acc=0.5616
Epoch  60: Loss=0.8012, Test Acc=0.5845
Epoch  70: Loss=0.7888, Test Acc=0.5845
Epoch  80: Loss=0.7636, Test Acc=0.6210
Epoch  90: Loss=0.7582, Test Acc=0.6119
Epoch  99: Loss=0.7509, Test Acc=0.6164
Best Epoch: 96 → Test Accuracy: 0.6256
Final Test Accuracy: 0.6164

Testing: dropout=0.1, lr=0.0001, hidden=[128, 64]
Epoch   0: Loss=1.1524, Test Acc=0.2603
Epoch  10: Loss=0.9603, Test Acc=0.5525
Epoch  20: Loss=0.9268, Test Acc=0.5525
Epoch  30: Loss=0.8767, Test Acc=0.5571
Epoch  40: Loss=0.8189, Test Acc=0.5616
Epoch  50: Loss=0.8207, Test Acc=0.5845
Epoch  60: Loss=0.7775, Test Acc=0.6027
Epoch  70: Loss=0.7310, Test Acc=0.6347
Epoch  80: Loss=0.7307, Test Acc=0.6073
Epoch  90: Loss=0.7249, Test Acc

,dropout,lr,hidden,acc
2,0.1,0.0001,"[256, 128]",0.657534
8,0.1,0.0010,"[256, 128]",0.648402
17,0.3,0.0010,"[256, 128]",0.648402
14,0.3,0.0005,"[256, 128]",0.643836
13,0.3,0.0005,"[128, 64]",0.643836
4,0.1,0.0005,"[128, 64]",0.639269
15,0.3,0.0010,"[64, 32]",0.639269
11,0.3,0.0001,"[256, 128]",0.639269
3,0.1,0.0005,"[64, 32]",0.639269
16,0.3,0.0010,"[128, 64]",0.634703


Best configuration: dropout           0.1
lr             0.0001
hidden     [256, 128]
acc          0.657534
Name: 2, dtype: object
Epoch   0: Loss=1.0750, Test Acc=0.5251
Epoch  10: Loss=0.9232, Test Acc=0.5525
Epoch  20: Loss=0.8529, Test Acc=0.5571
Epoch  30: Loss=0.8004, Test Acc=0.5799
Epoch  40: Loss=0.7477, Test Acc=0.6027
Epoch  50: Loss=0.7473, Test Acc=0.6119
Epoch  60: Loss=0.6849, Test Acc=0.6301
Epoch  70: Loss=0.6738, Test Acc=0.6210
Epoch  80: Loss=0.6564, Test Acc=0.6347
Epoch  90: Loss=0.5934, Test Acc=0.6347
Epoch 100: Loss=0.5717, Test Acc=0.6575
Epoch 110: Loss=0.5477, Test Acc=0.6484
Epoch 120: Loss=0.5098, Test Acc=0.6347
Epoch 130: Loss=0.4645, Test Acc=0.6301
Epoch 140: Loss=0.4853, Test Acc=0.6347
Epoch 149: Loss=0.4191, Test Acc=0.6484
Best Epoch: 98 → Test Accuracy: 0.6575
